In [ ]:
!pip install langchain sentence-transformers chromadb langchain-community

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

texto_bruto = """S
A empresa Alpha lucrou 5 milhões em 2023. A nova política de férias permite 30 dias.
O setor de vendas bateu a meta no primeiro trimestre.
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " "]
)

textos_divididos = text_splitter.split_text(texto_bruto)

documentos_prontos = []
for texto in textos_divididos:
    doc = Document(
        page_content=texto,
        metadata={"fonte": "relatorio_anual", "ano": 2023}
    )
    documentos_prontos.append(doc)

print(f"Foram criados {len(documentos_prontos)} pedaços de contexto estruturado.")

In [ ]:
from sentence_transformers import CrossEncoder

pergunta_usuario = "Qual foi o lucro da empresa Alpha?"

documentos_recuperados = [
    "A empresa Beta teve um prejuízo em 2023.",
    "A empresa Alpha lucrou 5 milhões em 2023.",
    "O setor de vendas da Alpha bateu a meta.",
    "Lucro é um conceito contábil importante."
]

modelo_rerank = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

pares_para_avaliar = [[pergunta_usuario, doc] for doc in documentos_recuperados]

notas = modelo_rerank.predict(pares_para_avaliar)

resultados_com_nota = list(zip(notas, documentos_recuperados))
resultados_ordenados = sorted(resultados_com_nota, key=lambda x: x[0], reverse=True)

top_n = 2
melhores_contextos = [doc for nota, doc in resultados_ordenados[:top_n]]

print("\n--- Contexto Final Refinado para enviar à LLM ---")
for i, contexto in enumerate(melhores_contextos):
    print(f"Top {i+1}: {contexto}")

In [ ]:
documentos_da_aula = [
    "O projeto da N2 vale 10 pontos e deve ser entregue até o final do semestre.",
    "Técnicas de RAG Avançado incluem pré e pós-recuperação para mitigar alucinações.",
    "O banco de dados vetorial armazena embeddings gerados a partir dos chunks."
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)

documentos_preparados = []
for i, texto in enumerate(documentos_da_aula):
    doc = Document(
        page_content=texto,
        metadata={"id_segmento": i, "materia": "IA_Avancada"}
    )
    documentos_preparados.append(doc)

In [ ]:
modelo_rerank = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modelo_embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

banco_vetorial = Chroma.from_documents(documentos_preparados, modelo_embedding)

pergunta = "Como mitigar alucinações no RAG?"
documentos_recuperados = banco_vetorial.similarity_search(pergunta, k=3)

textos_recuperados = [doc.page_content for doc in documentos_recuperados]

print("Busca no banco vetorial concluída!")

pares = [[pergunta, texto] for texto in textos_recuperados]
notas = modelo_rerank.predict(pares)

resultados_finais = sorted(list(zip(notas, textos_recuperados)), key=lambda x: x[0], reverse=True)

contexto_perfeito = resultados_finais[0][1]
print(f"Melhor contexto encontrado após o re-ranqueamento:\n-> {contexto_perfeito}")

In [ ]:
prompt_final = f"""
Você é um assistente de IA especializado. Responda à pergunta do usuário utilizando estritamente o contexto fornecido abaixo.

Contexto: {contexto_perfeito}
Pergunta: {pergunta}

Resposta:
"""
print("\n--- PROMPT PRONTO PARA ENVIAR À LLM ---")
print(prompt_final)